# 🚀 Thai BERT LoRA — Full Pipeline Demo บน Colab (6 ขั้นจาก README)
---
Notebook นี้แปลงมาจากหัวข้อ **"วิธีการรันล่าสุด"** ใน `README.md` (6 คำสั่ง) ให้รันได้จริงบน Google Colab

| # | ขั้นตอน | คำสั่งใน README | Cell |
|---|---|---|---|
| 1 | ขยายคำศัพท์ (Vocabulary Expansion) | `python scripts/extend_tokenizer.py` | 1️⃣ |
| 2 | เทรน Masked Language Modeling (MLM) | `python scripts/train_lora_mlm.py` | 2️⃣ |
| 3 | ทดสอบผลลัพธ์ (Inference Test) | `python scripts/test_inference.py` | 3️⃣ |
| 4 | รวมร่างโมเดล (Merge LoRA) | `python scripts/merge_lora.py` | 4️⃣ |
| 5 | เทรน SapBERT NER | `python scripts/train_sapbert_ner.py` | 5️⃣ |
| 6 | ทดสอบ NER Inference | `python scripts/inference_ner.py --model ... --text ...` | 6️⃣ |

## 🎛️ โหมดการรัน 2 แบบ
- **QUICK_DEMO = True** (แนะนำครั้งแรก): สุ่มตัวอย่างข้อมูล ~30,000 ประโยคมาเทรน → รันครบ 6 ขั้นใน **~30–60 นาที** (T4 GPU) เป้าหมายคือเห็น pipeline ทำงานครบวงจร (คะแนนจะไม่สวยเท่าของจริง)
- **QUICK_DEMO = False** (FULL): ใช้ข้อมูลเต็ม 2.46 ล้านประโยค → เฉพาะขั้น 2 ใช้เวลา **8–15 ชม.** (Colab ฟรีมีโอกาสหลุดกลางคัน — แนะนำ Colab Pro และดูหัวข้อ Troubleshooting ท้าย notebook เรื่อง `--resume`)

## 📦 สิ่งที่ต้องเตรียมล่วงหน้า (อัปโหลดขึ้น Google Drive)
```
MyDrive/
├── thai_bert_lora.zip                 ← จำเป็น: โปรเจกต์ zip (~30 MB — คำสั่งด้านล่าง)
├── SapBERTThaiMLM_CRF/                ← (แนะนำ) โมเดลต้นฉบับจากเครื่อง local (~440 MB)
│                                        ไม่มีก็ได้ → notebook จะโหลด cambridgeltl/SapBERT จาก HF แทน
└── thai_bert_lora/data/processed/     ← (แนะนำ) train.jsonl + validation.jsonl + test.jsonl (~1.1 GB)
                                         ไม่มีก็ได้ → QUICK_DEMO จะสร้าง corpus จาก lexitron_clean.jsonl
```

**คำสั่ง zip โปรเจกต์ (รันใน PowerShell หรือ cmd บน Windows — ใช้ `tar` ที่มากับ Windows 10/11):**
```powershell
cd I:\NECTEC\thai_bert_lora
tar -a -cf thai_bert_lora.zip scripts src configs requirements.txt data/interim/lexitron_clean.jsonl data/processed/ner_bio_th.jsonl data/processed/ner_bio_en.jsonl data/processed/ner_label_schema.json
```
> ⚠️ **ไม่ควรใช้ `Compress-Archive`** — Windows PowerShell 5.1 จะ (1) วางไฟล์ที่ระบุเป็นรายไฟล์ (เช่น `data\interim\lexitron_clean.jsonl`) ไว้ที่ **root ของ zip** ทำให้โครงสร้างโฟลเดอร์หาย และ (2) เก็บ path ด้วย backslash ซึ่งแตกบน Linux เพี้ยน — คำสั่ง `tar` ด้านบนสร้าง zip ที่ path ถูกต้องเสมอ (และ cell แตก zip ด้านล่างรองรับ zip ทุกแบบเผื่อไว้แล้ว)

⚠️ **ห้าม zip `.venv/` และ `outputs/`** — ใหญ่มากและไม่จำเป็น (notebook เทรนใหม่บน Colab ทั้งหมด)

> 💡 ถ้าไม่ได้เอาชุดข้อมูลใหญ่ขึ้น Drive — QUICK_DEMO จะสร้าง corpus เองจากพจนานุกรม Lexitron ซึ่ง pipeline ยังรันได้ครบ แต่ผลทำนาย [MASK] จะเอียงไปทางตัวเลข/สัญลักษณ์ (เป็นการ recreate ปัญหา Dictionary Overfitting ของ Phase 2 ในโปรเจกต์พอดี 😄)

## 0️⃣ เตรียมสภาพแวดล้อม (Setup)
ติดตั้งไลบรารี + ตรวจสอบ GPU (แนะนำ Runtime → Change runtime type → **T4 GPU**)

In [ ]:
# Pin เวอร์ชันไลบรารีตาม venv ของโปรเจกต์ (transformers 5.16.1 บนเครื่อง local) กันพฤติกรรมต่างกัน
# torch ไม่ pin — ใช้ตัวที่ Colab เตรียมไว้ให้เพราะผูกกับ CUDA driver
!pip install -q transformers==5.16.1 tokenizers==0.23.2 peft==0.20.0 datasets==5.0.1 accelerate==1.14.0 safetensors==0.8.0 sentencepiece seqeval pyyaml

import torch, transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {props.name} ({props.total_memory/1e9:.0f} GB)")
else:
    print("⚠️ ไม่พบ GPU — ไปที่ Runtime > Change runtime type > T4 GPU (เทรนบน CPU จะช้ามาก)")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ======== 👇 แก้ไขค่าตรงนี้ให้ตรงกับ Drive ของคุณ ========
DRIVE          = "/content/drive/MyDrive"
PROJECT_ZIP    = f"{DRIVE}/thai_bert_lora.zip"        # zip ของโปรเจกต์ (scripts/src/configs/data ขนาดเล็ก)
PROJECT_DIR    = "/content/thai_bert_lora"            # รันจาก local disk ของ Colab (เร็วกว่า Drive มาก)
SOURCE_MODEL   = f"{DRIVE}/SapBERTThaiMLM_CRF"        # โมเดลต้นฉบับ — ถ้าไม่มีจะ fallback ไปโหลดจาก HF
DATA_DRIVE_DIR = f"{DRIVE}/thai_bert_lora/data"       # โฟลเดอร์ที่มี processed/train.jsonl (ข้อมูลเต็ม ~1.1GB)

QUICK_DEMO = True          # True = สุ่มข้อมูลเล็กมา demo / False = เทรนเต็ม 2.46M ประโยค (8–15 ชม.)
DEMO_TRAIN_LINES = 30_000
DEMO_VAL_LINES   = 5_000
DEMO_TEST_LINES  = 5_000

In [ ]:
# แตก zip โปรเจกต์ลง local disk แล้วเข้าไปในโฟลเดอร์โปรเจกต์
# รองรับทั้ง zip จาก tar, zip เก่าของ PowerShell (entry เป็น backslash), และ zip ที่มีโฟลเดอร์ครอบ
import os, shutil, zipfile

if not os.path.exists(PROJECT_ZIP):
    raise SystemExit(f"❌ ไม่พบ {PROJECT_ZIP} — zip โปรเจกต์แล้วอัปโหลดขึ้น Drive ก่อน (ดูคำสั่ง zip ใน cell แรก)")

# หมายเหตุ: ไม่ rmtree โฟลเดอร์เดิม — แตกทับเฉพาะไฟล์โค้ด และเก็บ outputs/ เดิมไว้ใช้ --resume
os.makedirs(PROJECT_DIR, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP) as z:
    raw   = z.namelist()
    names = [n.replace("\\", "/") for n in raw]   # PowerShell 5.1 เก็บ entry ด้วย backslash
    tops  = {n.split("/")[0] for n in names if n}
    wrapper = ""
    if len(tops) == 1:                             # zip มีโฟลเดอร์ครอบชั้นเดียว → ตัดออก
        only = next(iter(tops))
        if any(n.startswith(only + "/") for n in names):
            wrapper = only + "/"
    n_files = 0
    for orig, norm in zip(raw, names):
        rel = norm[len(wrapper):] if wrapper else norm
        parts = rel.split("/")
        if not rel or rel.endswith("/") or ".." in parts:
            continue
        dest = os.path.join(PROJECT_DIR, *parts)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with z.open(orig) as src_f, open(dest, "wb") as dst_f:
            shutil.copyfileobj(src_f, dst_f)
        n_files += 1

os.chdir(PROJECT_DIR)
print(f"📂 แตกไฟล์ {n_files} รายการ (โฟลเดอร์ครอบ: {wrapper or 'ไม่มี'})")

required = [
    "scripts/extend_tokenizer.py", "scripts/train_lora_mlm.py", "scripts/test_inference.py",
    "scripts/merge_lora.py", "scripts/train_sapbert_ner.py", "scripts/inference_ner.py",
    "configs/model_config.yaml", "configs/lora_config.yaml", "configs/training_config.yaml",
    "configs/data_config.yaml", "configs/sapbert_ner_config.yaml",
    "src/models/lora.py",
    "data/processed/ner_bio_th.jsonl", "data/processed/ner_label_schema.json",
    "data/interim/lexitron_clean.jsonl",
]
missing = [p for p in required if not os.path.exists(p)]
assert not missing, (
    f"❌ zip ขาดไฟล์: {missing}\n"
    "   ถ้าไฟล์ data มากองอยู่แบบไม่มีโฟลเดอร์ (เช่น 'data\\interim\\lexitron_clean.jsonl' เป็นชื่อไฟล์เดียว)\n"
    "   แปลว่า zip ถูกสร้างด้วย Compress-Archive — ให้สร้างใหม่ด้วยคำสั่ง tar ใน cell แรก"
)
print("✅ โปรเจกต์พร้อมที่", os.getcwd())

In [ ]:
# เตรียมโมเดลต้นฉบับ (source model) สำหรับ Step 1
# ตัวเลือก A: คัดลอก SapBERTThaiMLM_CRF จาก Drive (เหมือนรันบนเครื่อง local)
# ตัวเลือก B: โหลด cambridgeltl/SapBERT-from-PubMedBERT-fulltext จาก HuggingFace (ต้นฉบับอังกฤษล้วน)
import os, shutil

SRC_MODEL = "/content/models/sapbert_source"

if SOURCE_MODEL and os.path.exists(SOURCE_MODEL):
    print(f"📦 คัดลอกโมเดลต้นฉบับจาก Drive: {SOURCE_MODEL}")
    shutil.copytree(
        SOURCE_MODEL, SRC_MODEL,
        dirs_exist_ok=True,  # รัน cell ซ้ำได้โดยไม่ error
        ignore=shutil.ignore_patterns("checkpoint-*", "optimizer.pt", "scheduler.pt", "rng_state*.pth"),
    )
else:
    if SOURCE_MODEL:
        print(f"⚠️ ไม่พบ {SOURCE_MODEL} ใน Drive — fallback โหลดจาก HuggingFace")
    from huggingface_hub import snapshot_download
    snapshot_download("cambridgeltl/SapBERT-from-PubMedBERT-fulltext", local_dir=SRC_MODEL)
    print("✅ โหลด cambridgeltl/SapBERT-from-PubMedBERT-fulltext แล้ว")

print("SRC_MODEL =", SRC_MODEL)
print(os.listdir(SRC_MODEL))

## 🧩 เตรียมข้อมูล + Config สำหรับ Colab
- **QUICK_DEMO**: สุ่ม ~30,000 ประโยคจาก `train.jsonl` ตัวจริง (ถ้ามีใน Drive) → `data/processed_demo/` — ถ้าไม่มีจะใช้พจนานุกรม `lexitron_clean.jsonl` สร้าง corpus ให้แทน
- **FULL**: คัดลอกข้อมูลเต็มจาก Drive มาไว้ local disk (I/O เร็วกว่า)
- สร้าง `configs/data_config_colab.yaml` + `configs/training_config_colab.yaml` ให้ script เดิมของโปรเจกต์อ่านได้โดยไม่ต้องแก้โค้ด

In [ ]:
import json, os, random, shutil, yaml

random.seed(42)
DEMO_DIR = "data/processed_demo"

def read_jsonl(path, limit=None):
    out = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

drive_processed = os.path.join(DATA_DRIVE_DIR, "processed")
has_drive_data  = os.path.exists(os.path.join(drive_processed, "train.jsonl"))
has_local_data  = os.path.exists("data/processed/train.jsonl")

if QUICK_DEMO:
    os.makedirs(DEMO_DIR, exist_ok=True)
    if has_drive_data or has_local_data:
        src = drive_processed if has_drive_data else "data/processed"
        print(f"✂️ Subsample ข้อมูลจริงจาก {src}")
        for name, n in [("train.jsonl", DEMO_TRAIN_LINES),
                        ("validation.jsonl", DEMO_VAL_LINES),
                        ("test.jsonl", DEMO_TEST_LINES)]:
            recs = read_jsonl(os.path.join(src, name), limit=n * 3)
            random.shuffle(recs)
            recs = recs[:n]
            write_jsonl(os.path.join(DEMO_DIR, name), recs)
            print(f"   {name}: {len(recs):,} ประโยค")
    else:
        print("⚠️ ไม่พบ corpus เต็ม — สร้าง demo corpus จาก lexitron_clean.jsonl (พจนานุกรมล้วน)")
        lex = read_jsonl("data/interim/lexitron_clean.jsonl")
        sents = []
        for e in lex:
            if e.get("definition"):
                sents.append(f'{e["word"]} หมายถึง {e["definition"]}')
            for ex in (e.get("examples") or [])[:1]:
                if ex:
                    sents.append(ex)
        random.shuffle(sents)
        write_jsonl(f"{DEMO_DIR}/train.jsonl",      [{"text": s} for s in sents[:DEMO_TRAIN_LINES]])
        write_jsonl(f"{DEMO_DIR}/validation.jsonl", [{"text": s} for s in sents[DEMO_TRAIN_LINES:DEMO_TRAIN_LINES + DEMO_VAL_LINES]])
        write_jsonl(f"{DEMO_DIR}/test.jsonl",       [{"text": s} for s in sents[-DEMO_TEST_LINES:]])
        print("   (คาดหวังได้ว่าการทาย [MASK] จะเอียงไปทางรูปแบบพจนานุกรม — เหมือน Phase 2 ของโปรเจกต์)")
    processed_dir = DEMO_DIR
else:
    if has_drive_data and not has_local_data:
        print(f"📦 คัดลอกข้อมูลเต็มจาก Drive → data/processed (~1.1GB อาจใช้เวลาหลายนาที)")
        os.makedirs("data/processed", exist_ok=True)
        for name in ["train.jsonl", "validation.jsonl", "test.jsonl"]:
            shutil.copy(os.path.join(drive_processed, name), f"data/processed/{name}")
    assert os.path.exists("data/processed/train.jsonl"), "❌ โหมด FULL ต้องมี train.jsonl (zip มาด้วย หรือใส่ใน Drive)"
    processed_dir = "data/processed"

# ── สร้าง data config สำหรับ Colab ──
cfg = yaml.safe_load(open("configs/data_config.yaml", encoding="utf-8"))
cfg["data"]["processed_dir"] = processed_dir
yaml.safe_dump(cfg, open("configs/data_config_colab.yaml", "w", encoding="utf-8"), allow_unicode=True)

# ── สร้าง training config สำหรับ Colab ──
tcfg = yaml.safe_load(open("configs/training_config.yaml", encoding="utf-8"))
if QUICK_DEMO:
    t = tcfg["training"]
    t.update({
        "per_device_train_batch_size": 8,      # ลดจาก 16 กัน VRAM T4 ไม่พอ (มี modules_to_save)
        "gradient_accumulation_steps": 4,      # effective batch = 32 เท่าเดิม
        "logging_steps": 25, "eval_steps": 200, "save_steps": 200, "save_total_limit": 1,
        "num_train_epochs": 1,
    })
yaml.safe_dump(tcfg, open("configs/training_config_colab.yaml", "w", encoding="utf-8"), allow_unicode=True)

print(f"✅ processed_dir = {processed_dir}")
print(f"✅ โหมด = {'QUICK_DEMO' if QUICK_DEMO else 'FULL'}")

---
## 1️⃣ Step 1 — ขยายคำศัพท์ (Vocabulary Expansion)
**README:** `python scripts/extend_tokenizer.py`

สิ่งที่เกิดขึ้นในขั้นนี้:
1. ฝึก SentencePiece BPE บน corpus ไทย → คำศัพท์ใหม่ ~6,000 คำ + ตัวอักษรไทยเดี่ยวเป็น fallback (กำจัด `[UNK]` 100%)
2. **Smart Init:** embedding ของคำไทยใหม่ = ค่าเฉลี่ยของ subwords เดิม (แก้ Circular Initialization Bug แล้ว)
3. บันทึกเป็น `data/interim/extended_base/` — โมเดลต้นฉบับไม่ถูกแตะ (read-only)

บน Colab เราใส่ `--source-model` (ชี้ไปที่โมเดลที่เตรียมไว้ใน cell ก่อนหน้า) และ `--data-config` ฉบับ Colab — นอกนั้นเหมือนคำสั่งใน README ทุกอย่าง

⏱️ QUICK: ~2–5 นาที | FULL: ~15–40 นาที (ฝึก SPM บน 2.4M ประโยค)

In [ ]:
!python scripts/extend_tokenizer.py --source-model "{SRC_MODEL}" --data-config configs/data_config_colab.yaml

print("\n✅ ผลลัพธ์:", os.listdir("data/interim/extended_base"))

## 2️⃣ Step 2 — เทรน Masked Language Modeling (MLM)
**README:** `python scripts/train_lora_mlm.py`

- LoRA r=32, alpha=64 บน `query/key/value/dense` + **`modules_to_save: [word_embeddings, cls.predictions]`** ← ตัวแก้ Frozen Embedding Bug (ถ้าขาดสองชั้นนี้ คำไทยใหม่จะถูกแช่แข็งเป็นค่าสุ่มตลอดการเทรน — โมเดลจะทายแต่ตัวเลข)
- Adapter ถูกเซฟที่ `outputs/adapters/lexitron_wiki_thai_lora_r32/`

⏱️ **QUICK:** ~940 steps → ~20–45 นาที (T4) | **FULL:** ~69,000 steps → **8–15 ชม.** ⚠️ Colab ฟรีจะหลุดก่อนจบ — ถ้าหลุดให้รัน cell นี้ใหม่ด้วย `--resume outputs/checkpoints/checkpoint-XXXX`

📊 ผลอ้างอิง (FULL บนเครื่อง local): MLM Loss 2.9584 / Perplexity **19.27** — โหมด QUICK จะได้เลขสูงกว่านี้มากเป็นเรื่องปกติ

In [ ]:
!python scripts/train_lora_mlm.py --data-config configs/data_config_colab.yaml --training-config configs/training_config_colab.yaml

## 3️⃣ Step 3 — ทดสอบผลลัพธ์ (MLM Inference Test)
**README:** `python scripts/test_inference.py`

ทดสอบเติมคำ `[MASK]` — ⚠️ ณ จุดนี้ยังไม่มี merged model Script จะทดสอบเฉพาะ **base model (extended_base ที่ยังไม่เทรน)** → คำไทยใหม่ embedding ยังเป็นค่า smart-init ทายคำไทยยังไม่แม่น **เป็นเรื่องปกติ** — ของจริงดูอีกครั้งหลัง merge (Step 4)

In [ ]:
!python scripts/test_inference.py

## 4️⃣ Step 4 — รวมร่างโมเดล (Merge LoRA)
**README:** `python scripts/merge_lora.py`

หลอม adapter เข้า base ตามสูตร `W_merged = W_base + (α/r)(B×A)` → บันทึกเป็น `outputs/merged_model_N` (auto-increment ไม่เขียนทับของเดิม)

In [ ]:
!python scripts/merge_lora.py

# หา merged model ล่าสุดที่เพิ่งสร้าง
import glob, os
cands = [p for p in glob.glob("outputs/merged_model_*")
         if p.rsplit("_", 1)[-1].isdigit() and os.path.exists(os.path.join(p, "config.json"))]
MERGED_MODEL = max(cands, key=lambda p: int(p.rsplit("_", 1)[-1]))
print("\n✅ MERGED_MODEL =", MERGED_MODEL)

In [ ]:
# ทดสอบซ้ำด้วย merged model — รอบนี้ top-5 ต้องเริ่มเป็นคำไทย (ถ้าเทรนพอ)
!python scripts/test_inference.py --merged "{MERGED_MODEL}"

## 5️⃣ Step 5 — เทรน SapBERT NER
**README:** `python scripts/train_sapbert_ner.py`

- Token Classification: DRUG / DISEASE / SYMPTOM (BIO, 7 labels) บนข้อมูลฉลากยาไทย 44 ประโยค (`data/processed/ner_bio_th.jsonl` — ไฟล์เล็ก อยู่ใน zip แล้ว)
- LoRA r=16, alpha=32 (`TaskType.TOKEN_CLS`), 100 epochs, วัด entity-level F1 ด้วย seqeval
- เทรนเสร็จ script จะ **auto-merge** ให้เลย → `outputs/merged_model_N_ner_M`

⚠️ เราใส่ `--base-model` ชัดเจน (ต่างจาก README) — เพราะ auto-detect ของ script อาจไปเลือกโมเดล NER เก่ามาเทรนซ้อน (NER ซ้อน NER) ถ้ามีหลายโฟลเดอร์ใน outputs/

⏱️ ~3–10 นาที (GPU) | 📊 ผลอ้างอิงบนเครื่อง local: F1 = 0.7714 (base = merged_model_1) และ 0.8000 (รอบสอง) | baseline WangchanBERTa = 0.8269

In [ ]:
!python scripts/train_sapbert_ner.py --base-model "{MERGED_MODEL}"

# หา NER model ที่เพิ่งเทรนเสร็จ
import glob, os
stem = os.path.basename(MERGED_MODEL)
runs = [p for p in glob.glob(f"outputs/{stem}_ner_*") if os.path.exists(os.path.join(p, "config.json"))]
NER_MODEL = max(runs, key=os.path.getmtime)
print("\n✅ NER_MODEL =", NER_MODEL)

## 6️⃣ Step 6 — ทดสอบ NER Inference
**README:** `python scripts/inference_ner.py --model outputs/merged_model_1_ner_base --text "..."`

บน Colab เราใช้โมเดลที่เพิ่งเทรนใน Step 5 (`NER_MODEL`) — ลองแก้ `--text` เป็นประโยคฉลากยาอื่นได้เลย

ผลลัพธ์ที่คาดหวัง (โมเดลที่เทรนเต็มบนเครื่อง local):
```
[DRUG    ]  'อะบิราเทอโรน'        (pos 0–12)
[DISEASE ]  'มะเร็งต่อมลูกหมาก'    (pos 21–38)
[SYMPTOM ]  'แพร่กระจาย'          (pos 49–59)
```

In [ ]:
!python scripts/inference_ner.py --model "{NER_MODEL}" --text "อะบิราเทอโรน ใช้รักษามะเร็งต่อมลูกหมาก โดยมีอาการแพร่กระจาย"

In [ ]:
# ทดสอบประโยคอื่นเพิ่มเติม (interactive ไม่ได้บน Colab — ใช้ --text ทีละประโยค)
!python scripts/inference_ner.py --model "{NER_MODEL}" --text "อะดาลิมูแมบ ใช้รักษาข้ออักเสบรูมาตอยด์ มีอาการอักเสบ"
!python scripts/inference_ner.py --model "{NER_MODEL}" --text "พาราเซตามอล บรรเทาอาการปวดและลดไข้"

## 💾 (Optional) เก็บผลลัพธ์กลับ Google Drive
Colab ลบเครื่องเมื่อจบ session — ถ้าอยากเก็บ adapter / merged model ไว้ ให้รัน cell นี้

In [ ]:
import os, shutil

SAVE_DIR = f"{DRIVE}/thai_bert_lora_outputs_colab"
os.makedirs(SAVE_DIR, exist_ok=True)

for p in ["outputs/adapters/lexitron_wiki_thai_lora_r32", MERGED_MODEL, NER_MODEL]:
    if os.path.exists(p):
        dst = os.path.join(SAVE_DIR, os.path.basename(p))
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(p, dst)
        print("💾 saved:", dst)
    else:
        print("⚠️ ไม่พบ (ข้าม):", p)

---
## 📊 สรุปผลที่คาดหวัง & Troubleshooting

| ขั้น | QUICK_DEMO (30k ประโยค) | FULL (2.46M ประโยค — ผลอ้างอิงจากเครื่อง local) |
|---|---|---|
| 2️⃣ MLM | loss ลดแต่ยังไม่ converge | Loss 2.9584 / **PPL 19.27** |
| 3️⃣/4️⃣ ทาย [MASK] | อาจได้คำไทยบ้าง/ยังไม่ดี | top-5 เป็นคำไทยตรงบริบท (เช่น "รักษา", "ประชาชน") |
| 5️⃣ NER F1 | ต่ำ–ปานกลาง (แล้วแต่คุณภาพ MLM) | **0.7714 → 0.8000** (baseline WangchanBERTa = 0.8269) |
| 6️⃣ NER inference | จับ entity ได้บ้าง | จับ DRUG/DISEASE/SYMPTOM ถูกต้องตามตัวอย่าง |

**ปัญหาที่พบบ่อย:**
- **CUDA out of memory** → ลด `per_device_train_batch_size` ใน `configs/training_config_colab.yaml` (เช่น 8 → 4) แล้วรัน Step 2 ใหม่
- **Colab หลุดกลาง Step 2 (FULL)** → checkpoint อยู่ที่ `/content/thai_bert_lora/outputs/checkpoints/` (**ไม่ได้อยู่ใน Drive อัตโนมัติ**)
  - ถ้า reconnect แล้วได้ VM เดิม (`/content` ยังอยู่) → รัน cell mount Drive แล้วรัน Step 2 ต่อด้วย `--resume outputs/checkpoints/checkpoint-XXXX` ได้เลย (cell แตก zip ไม่ลบ `outputs/` ทิ้ง)
  - ถ้าได้ VM ใหม่ → `/content` หายหมด ต้องเริ่ม Step 2 ใหม่ — กันไว้โดย copy checkpoint ขึ้น Drive เป็นระยะจาก cell แยก เช่น `!cp -r outputs/checkpoints "{DRIVE}/thai_bert_lora_ckpt"` (แนะนำ Colab Pro)
- **Step 1 แจ้งหา corpus ไม่เจอ** → ต้องมี `train.jsonl` + `validation.jsonl` + `test.jsonl` ใน processed_dir ที่ config ชี้ (cell เตรียมข้อมูลสร้างให้แล้ว)
- **Step 6 ไม่เจอโมเดล** → เช็คว่า `NER_MODEL` ใน cell ก่อนหน้าชี้ไปโฟลเดอร์ที่มี `config.json` + `label_schema.json`

**หมายเหตุ:** notebook นี้รัน script จริงของโปรเจกต์ (`scripts/*.py` + `src/`) เหมือนคำสั่งใน README — ต่างแค่ (1) path override สำหรับ Colab (`--source-model`, `--data-config`, `--training-config`, `--base-model`, `--model`) และ (2) pin เวอร์ชันไลบรารีตาม venv ของเครื่อง local ถ้าแก้โค้ดในโปรเจกต์ ผลบน Colab จะเปลี่ยนตาม